In [1]:
##modules
#%matplotlib widget
#%matplotlib inline
#
#%matplotlib qt
import mne
import numpy as np
import matplotlib
# Establecer un backend interactivo, como 'Qt5Agg', 'GTK3Agg', etc.
# Esto depende de los backends disponibles en tu sistema.

import matplotlib.pyplot as plt

matplotlib.use('TkAgg')  # Asegúrate de que este backend está instalado.

import pandas as pd 
import os
import sys

from mne.preprocessing import ICA, corrmap, create_ecg_epochs, create_eog_epochs

from os.path import join as pathjoin
from time import time

from pathlib import Path

from autoreject import AutoReject

# aplicar la acf EN epochs
from statsmodels.tsa.stattools import acf

import numpy as np

import pandas as pd


from joblib import Parallel, delayed

import pickle
sys.path.append(r"G:\apuntes_mne\code_MOUS\scripts_functions_created")

# Ahora importa la función
from print5 import print5

import re


In [2]:

try:
    # Si se ejecuta como SCRIPT .py: usar __file__
    sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
except NameError:
    # Si se ejecuta como NOTEBOOK Jupyter: usar path relativo
    sys.path.append("..")  # sube un nivel desde la carpeta actual del notebook


# --- Configuración dinámica de rutas ---
from get_paths_SELF import get_paths_SELF

# Parámetros editables
disco = "g"
modality="visual"
layer_script = "event"
subj= "s01b"


# Generar variables automáticamente
path_dict = get_paths_SELF(disco=disco, modality=modality,layer_script=layer_script,  subj=subj)
globals().update(path_dict)

# Mostrar todos los paths generados
print("\n📁 Rutas generadas:")
for k, v in path_dict.items():
    print(f"{k:<20} → {v}")


✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_preproc\preproc_event\epochs_event
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_preproc\preproc_event\ICA_event
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_preproc\preproc_event\epochs_clean_event
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_preproc\preproc_event\epochs_matlab_event
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_preproc\preproc_event\evoked_event
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\channels_structure
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_source\source_event
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_source\source_event\raw_hsp
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_source\source_event\fwd
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_source\source_event\inverse
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_analysis\analysis_event\acw_event
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_analysis\analysis_eve

In [3]:
filtering=True
if filtering==True:
    lfreq=1
    hfreq=40
    filter_name= f"filt_{lfreq}-{hfreq}"
    print(f"Filtrado aplicado: {lfreq}-{hfreq} Hz")
elif filtering==False:
    print("No se ha aplicado filtrado.")

Filtrado aplicado: 1-40 Hz


In [4]:
#epochs

subjects_BV = []

# Busca archivos .vhdr dentro de export_generic_data
for archivo in export_generic_data.glob("*.vhdr"):
    nombre = archivo.stem  # sin la extensión .vhdr

    # Ejemplo: s01b_vis_c_BV_mne -> queremos s01b
    sujeto = nombre.split("_")[0]

    subjects_BV.append(sujeto)


subjects_BV = sorted(set(subjects_BV), key=str.lower)
print(subjects_BV)


##tablas de canales

['s01b', 's02b', 's03b', 's04b', 's05b', 's06b', 's07b', 's08b', 's09b', 's10b', 's11b', 's12b', 's13b', 's14b', 's15b', 's16b', 's17b', 's18b', 'S19b', 'S20b', 'S21b', 'S22b', 'S23b', 'S24b', 'S25b', 'S26b', 'S27b', 'S28b', 'S29b', 'S30b', 'S31b', 's32b', 'S33b', 'S34b', 'S35b', 's36b']


In [5]:
# Dictionary to store subjects with missing events
incomplete_epochs = {}

# Expected emotional event codes
event_id_emoc = [
    14, 15, 16,
    24, 25, 26,
    34, 35, 36,
    44, 45, 46,
    54, 55, 56,
    64, 65, 66,
    74, 75, 76,
    84, 85, 86,
    94, 95, 96,
]

for subj in subjects_BV:

    # --- Find vhdr file flexibly ---
    files = list(export_generic_data.glob(f"{subj}*.vhdr"))

    if len(files) == 0:
        print(f"❌ No file found for {subj}")
        continue

    elif len(files) > 1:
        print(f"⚠️ Multiple files found for {subj}:")
        for f in files:
            print(f)
        continue

    file_path = files[0]

    print(f"📂 Loading: {file_path.name}")

    # Load data
    raw = mne.io.read_raw_brainvision(file_path, preload=True)

    # Set EOG channels
    raw.set_channel_types({
        "HEOG+": "eog",
        "HEOG": "eog",
        "VEOG+": "eog",
        "VEOG": "eog",
    })

    # Extract events
    events, event_id = mne.events_from_annotations(raw)

    # for name, code in event_id.items():
    #     n = (events[:, 2] == code).sum()
    #     print(f"{name}: {n}")



    # Get event codes actually present in this subject
    available_events = sorted(set(events[:, 2]))

    # Find missing expected events
    missing_events = [ev for ev in event_id_emoc if ev not in available_events]

    # Store missing events only if there are any
    if missing_events:
        incomplete_epochs[subj] = missing_events
        print(f"⚠️ {subj} is missing events: {missing_events}")

    # Crear epochs
    epochs_emoc = mne.Epochs(
        raw,
        events,
        event_id=event_id_emoc,
        tmin=0,
        tmax=6,
        baseline=None,
        preload=True,
        reject=None, flat=None, 
        proj=True, 
        decim=1, 
        reject_tmin=None, 
        reject_tmax=None, 
        detrend=None, 
        on_missing='ignore', 
        reject_by_annotation=False
        
        
    )


    epochs_self = mne.Epochs(
        raw,
        events,
        event_id=event_id_emoc,
        tmin=-2,
        tmax=0,
        baseline=None,
        preload=True,
        reject=None, flat=None, 
        proj=True, 
        decim=1, 
        reject_tmin=None, 
        reject_tmax=None, 
        detrend=None, 
        on_missing='ignore', 
        reject_by_annotation=False
    
    )

    epochs_full = mne.Epochs(
        raw,
        events,
        event_id=event_id_emoc,
        tmin=-2,
        tmax=6,
        baseline=None,
        preload=True,
        reject=None, flat=None, 
        proj=True, 
        decim=1, 
        reject_tmin=None, 
        reject_tmax=None, 
        detrend=None, 
        on_missing='ignore', 
        reject_by_annotation=False
    
    )


    print(epochs_emoc)

    print(epochs_self)

    print(epochs_full)


        # --- Save epochs ---
    fname_emoc = epochs_clean_path / f"{subj}_epochs_emoc-epo.fif"
    fname_self = epochs_clean_path / f"{subj}_epochs_self-epo.fif"
    fname_full = epochs_clean_path / f"{subj}_epochs_full-epo.fif"

    epochs_emoc.save(fname_emoc, overwrite=True)
    epochs_self.save(fname_self, overwrite=True)
    epochs_full.save(fname_full, overwrite=True)

    print(f"✅ Saved epochs for {subj}")

# Print summary of incomplete subjects
print("\n📋 Subjects with incomplete event sets:")
print(incomplete_epochs)

📂 Loading: s01b_vis_c_BV_mne.vhdr
Extracting parameters from g:\PROYECTO_SELF\SELF_visual\BRAIN_VISION_SELF\export_generic_data\s01b_vis_c_BV_mne.vhdr...
Setting channel info structure...
Reading 0 ... 470015  =      0.000 ...  1835.996 secs...


C:\Users\UCM\AppData\Local\Temp\ipykernel_11420\716927847.py:37: RuntimeWarning: No coordinate information found for channels ['HEOG+', 'HEOG', 'VEOG+', 'VEOG', 'M1', 'M2']. Setting channel types to misc. To avoid this warning, set channel types explicitly.
  raw = mne.io.read_raw_brainvision(file_path, preload=True)


Used Annotations descriptions: [np.str_('New Segment/'), np.str_('Stimulus/S  1'), np.str_('Stimulus/S  2'), np.str_('Stimulus/S  3'), np.str_('Stimulus/S  4'), np.str_('Stimulus/S  5'), np.str_('Stimulus/S  6'), np.str_('Stimulus/S  7'), np.str_('Stimulus/S  8'), np.str_('Stimulus/S  9'), np.str_('Stimulus/S 14'), np.str_('Stimulus/S 15'), np.str_('Stimulus/S 16'), np.str_('Stimulus/S 24'), np.str_('Stimulus/S 24_e'), np.str_('Stimulus/S 25'), np.str_('Stimulus/S 26'), np.str_('Stimulus/S 34'), np.str_('Stimulus/S 35'), np.str_('Stimulus/S 36'), np.str_('Stimulus/S 44'), np.str_('Stimulus/S 45'), np.str_('Stimulus/S 46'), np.str_('Stimulus/S 54'), np.str_('Stimulus/S 55'), np.str_('Stimulus/S 55_e'), np.str_('Stimulus/S 56'), np.str_('Stimulus/S 64'), np.str_('Stimulus/S 65'), np.str_('Stimulus/S 66'), np.str_('Stimulus/S 74'), np.str_('Stimulus/S 75'), np.str_('Stimulus/S 76'), np.str_('Stimulus/S 84'), np.str_('Stimulus/S 84_e'), np.str_('Stimulus/S 85'), np.str_('Stimulus/S 86'), n

C:\Users\UCM\AppData\Local\Temp\ipykernel_11420\716927847.py:37: RuntimeWarning: Not setting positions of 6 misc channels found in montage:
['HEOG+', 'HEOG', 'VEOG+', 'VEOG', 'M1', 'M2']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_brainvision(file_path, preload=True)
C:\Users\UCM\AppData\Local\Temp\ipykernel_11420\716927847.py:40: RuntimeWarning: The unit for channel(s) HEOG, HEOG+, VEOG, VEOG+ has changed from NA to V.
  raw.set_channel_types({


1 bad epochs dropped
Not setting metadata
213 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 213 events and 513 original time points ...
0 bad epochs dropped
Not setting metadata
213 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 213 events and 2049 original time points ...
1 bad epochs dropped
<Epochs |  212 events (all good), 0 – 6 s, baseline off, ~161.7 MB, data loaded,
 '14': 7
 '15': 9
 '16': 8
 '24': 7
 '25': 6
 '26': 9
 '34': 6
 '35': 7
 '36': 10
 '44': 10
 and 17 more events ...>
<Epochs |  213 events (all good), -2 – 0 s, baseline off, ~54.3 MB, data loaded,
 '14': 7
 '15': 9
 '16': 8
 '24': 7
 '25': 6
 '26': 9
 '34': 6
 '35': 7
 '36': 10
 '44': 10
 and 17 more events ...>
<Epochs |  212 events (all good), -2 – 6 s, baseline off, ~215.5 MB, data loaded,
 '14': 7
 '15': 9
 '16': 8
 '24': 7
 '25': 6
 '26': 9
 '34': 6
 '35': 7
 '36': 10
 '44':

...

1 bad epochs dropped
Not setting metadata
168 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 168 events and 513 original time points ...
0 bad epochs dropped
Not setting metadata
168 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 168 events and 2049 original time points ...
1 bad epochs dropped
<Epochs |  167 events (all good), 0 – 6 s, baseline off, ~127.4 MB, data loaded,
 '14': 8
 '15': 6
 '16': 6
 '24': 4
 '25': 5
 '26': 7
 '34': 5
 '35': 8
 '36': 8
 '44': 5
 and 17 more events ...>
<Epochs |  168 events (all good), -2 – 0 s, baseline off, ~42.8 MB, data loaded,
 '14': 8
 '15': 6
 '16': 6
 '24': 4
 '25': 5
 '26': 8
 '34': 5
 '35': 8
 '36': 8
 '44': 5
 and 17 more events ...>
<Epochs |  167 events (all good), -2 – 6 s, baseline off, ~169.8 MB, data loaded,
 '14': 8
 '15': 6
 '16': 6
 '24': 4
 '25': 5
 '26': 7
 '34': 5
 '35': 8
 '36': 8
 '44': 5
 a

C:\Users\UCM\AppData\Local\Temp\ipykernel_11420\716927847.py:37: RuntimeWarning: No coordinate information found for channels ['HEOG+', 'HEOG', 'VEOG+', 'VEOG', 'M1', 'M2']. Setting channel types to misc. To avoid this warning, set channel types explicitly.
  raw = mne.io.read_raw_brainvision(file_path, preload=True)


Used Annotations descriptions: [np.str_('New Segment/'), np.str_('Stimulus/S  1'), np.str_('Stimulus/S  2'), np.str_('Stimulus/S  3'), np.str_('Stimulus/S  4'), np.str_('Stimulus/S  5'), np.str_('Stimulus/S  6'), np.str_('Stimulus/S  7'), np.str_('Stimulus/S  8'), np.str_('Stimulus/S  9'), np.str_('Stimulus/S 14'), np.str_('Stimulus/S 15'), np.str_('Stimulus/S 16'), np.str_('Stimulus/S 24'), np.str_('Stimulus/S 25'), np.str_('Stimulus/S 26'), np.str_('Stimulus/S 34'), np.str_('Stimulus/S 35'), np.str_('Stimulus/S 36'), np.str_('Stimulus/S 44'), np.str_('Stimulus/S 45'), np.str_('Stimulus/S 46'), np.str_('Stimulus/S 54'), np.str_('Stimulus/S 55'), np.str_('Stimulus/S 56'), np.str_('Stimulus/S 64'), np.str_('Stimulus/S 65'), np.str_('Stimulus/S 66'), np.str_('Stimulus/S 74'), np.str_('Stimulus/S 75'), np.str_('Stimulus/S 76'), np.str_('Stimulus/S 84'), np.str_('Stimulus/S 85'), np.str_('Stimulus/S 86'), np.str_('Stimulus/S 94'), np.str_('Stimulus/S 95'), np.str_('Stimulus/S 96'), np.str_

C:\Users\UCM\AppData\Local\Temp\ipykernel_11420\716927847.py:37: RuntimeWarning: Not setting positions of 6 misc channels found in montage:
['HEOG+', 'HEOG', 'VEOG+', 'VEOG', 'M1', 'M2']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_brainvision(file_path, preload=True)
C:\Users\UCM\AppData\Local\Temp\ipykernel_11420\716927847.py:40: RuntimeWarning: The unit for channel(s) HEOG, HEOG+, VEOG, VEOG+ has changed from NA to V.
  raw.set_channel_types({


Not setting metadata
178 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 178 events and 513 original time points ...
0 bad epochs dropped
Not setting metadata
178 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 178 events and 2049 original time points ...
1 bad epochs dropped
<Epochs |  177 events (all good), 0 – 6 s, baseline off, ~135.0 MB, data loaded,
 '14': 3
 '15': 6
 '16': 10
 '24': 9
 '25': 3
 '26': 7
 '34': 5
 '35': 8
 '36': 8
 '44': 5
 and 17 more events ...>
<Epochs |  178 events (all good), -2 – 0 s, baseline off, ~45.4 MB, data loaded,
 '14': 3
 '15': 6
 '16': 10
 '24': 9
 '25': 3
 '26': 7
 '34': 5
 '35': 8
 '36': 8
 '44': 5
 and 17 more events ...>
<Epochs |  177 events (all good), -2 – 6 s, baseline off, ~179.9 MB, data loaded,
 '14': 3
 '15': 6
 '16': 10
 '24': 9
 '25': 3
 '26': 7
 '34': 5
 '35': 8
 '36': 8
 '44': 5
 and 17 more events 

In [6]:
# Subjects with incomplete event sets
subjects_incomplete = list(incomplete_epochs.keys())

# Subjects with complete event sets
subjects_complete = [s for s in subjects_BV if s not in subjects_incomplete]

print("\n✅ Subjects complete:")
print(subjects_complete)

print("\n⚠️ Subjects incomplete:")
print(subjects_incomplete)


✅ Subjects complete:
['s01b', 's02b', 's03b', 's04b', 's06b', 's07b', 's10b', 's11b', 's12b', 's13b', 's15b', 's16b', 'S19b', 'S20b', 'S21b', 'S22b', 'S23b', 'S24b', 'S25b', 'S26b', 'S27b', 'S28b', 'S29b', 'S30b', 'S31b', 's32b', 'S33b', 'S34b', 'S35b', 's36b']

⚠️ Subjects incomplete:
['s05b', 's08b', 's09b', 's14b', 's17b', 's18b']


In [7]:

# Save incomplete subjects (dictionary with missing events)
with open(os.path.join(output_analysis, "subjects_incomplete_epochs.pkl"), "wb") as f:
    pickle.dump(incomplete_epochs, f)

# Save list of incomplete subjects
with open(os.path.join(output_analysis, "subjects_incomplete_list.pkl"), "wb") as f:
    pickle.dump(subjects_incomplete, f)

# Save list of complete subjects
with open(os.path.join(output_analysis, "subjects_complete_list.pkl"), "wb") as f:
    pickle.dump(subjects_complete, f)

print("✅ Subject lists saved")

✅ Subject lists saved
